# AML Alert Prioritization — DnkCode

**Team 2ABB3C78 · WIUT Hackathon 2026 · FinTech / AI in Finance**

Predict the probability that an AML alert is escalated. Metric: ROC-AUC.

Design: `docs/superpowers/specs/2026-09-25-aml-alert-prioritization-design.md`  
Plan: `docs/superpowers/plans/2026-09-25-aml-alert-prioritization.md`

Every figure in the prose below is computed by a cell. Nothing is typed by hand.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd

from src import config
from src.data import load_signals, load_transactions
from src.features import build_features, prepare_transactions
from src.models import make_factory
from src.selection import load_selection
from src.site_data import compute
from src.validation import evaluate

## The alerts

In [ ]:
signals = load_signals(config.TRAIN_SIGNALS)
print('alerts:', len(signals))
print('escalation rate:', f"{signals[config.TARGET].mean():.4f}")
print('date range:', signals.signal_sanasi.min().date(), '->', signals.signal_sanasi.max().date())
signals.head()

## The transaction history

In [ ]:
tx_raw = load_transactions(config.TRAIN_TX)
print('raw transactions:', len(tx_raw))
tx = prepare_transactions(tx_raw, signals)
print('after dropping days_before < 0:', len(tx))
print('dropped:', len(tx_raw) - len(tx), f"({(len(tx_raw)-len(tx))/len(tx_raw):.5%})")
tx.days_before.describe()

### Why a few transactions look like they happen after their alert

`signal_sanasi` is a date, so it parses to midnight. A transaction on the alert's
own day therefore lands fractionally *after* it — the minimum `days_before` before
filtering is about −1.0 days, not −30. It is a timestamp artifact, not leakage from
the future, and the count above shows how small it is. We drop those rows rather
than reason about them per row.

The history is otherwise bounded to a fixed 180-day window ending at the alert.
There is no long-horizon behaviour to mine.

## Exploratory views

In [ ]:
tables = compute(signals, tx)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

t = tables['target']
axes[0,0].bar(t['outcome'], t['alerts'], color=['#4a8f79', '#d36a52'])
axes[0,0].set_title('Target distribution')

w = tables['weekly_volume']
axes[0,1].plot(w['tranzaksiya_vaqti'], w['transactions'], color='#3f6f8f')
axes[0,1].set_title('Weekly transaction volume')

ty = tables['types']
axes[1,0].bar(ty['type'], ty['transactions'], color='#7b6f9f')
axes[1,0].set_title('Transaction type')
axes[1,0].tick_params(axis='x', rotation=20)

h = tables['days_before_hist']
for outcome, group in h.groupby('eskalatsiya'):
    share = group['transactions'] / group['transactions'].sum()
    axes[1,1].plot(group['bucket'], share, label=outcome)
axes[1,1].set_title('Activity before the alert')
axes[1,1].set_xlabel('Days before')
axes[1,1].legend()

plt.tight_layout()

In [ ]:
tables['amount_by_outcome']

In [ ]:
tables['volume_by_outcome']

## The finding that shaped the model

We measured each feature family by repeated cross-validation instead of assuming it
helped. The table below is the actual forward-selection trace: every family was
offered at every round, and only the one that improved `mean − std` was accepted.

Accuracy peaks on a small set. Several families score at or below chance on their
own, and adding them costs accuracy. On this dataset the binding constraint is
variance, not model capacity.

In [ ]:
summary = json.loads((config.EXPERIMENTS / 'summary.json').read_text())
history = pd.DataFrame(summary['selection_history'])
history['round'] = history.groupby('family').cumcount() + 1
history[['round', 'family', 'n_columns', 'mean', 'std', 'score', 'accepted']]

In [ ]:
first_round = history[history['round'] == 1].sort_values('score', ascending=False)
first_round[['family', 'n_columns', 'mean', 'std', 'score']].reset_index(drop=True)

In [ ]:
print('families kept:', summary['families'])
print('columns kept:', summary['n_columns'])
print('per-model score (mean - std):')
for name, score in summary['model_scores'].items():
    print(f'  {name}: {score:.5f}')
print('ensemble weighting:', summary['weighting'], summary['weights'])
print('ensemble OOF ROC-AUC:', round(summary['ensemble_oof_auc'], 5))

## Reproducing the reported cross-validation score

The selection and the hyperparameter search are recorded in `experiments/log.csv`
and `experiments/summary.json`. This cell rebuilds the frozen feature set and
re-runs the harness on it, so the number below is regenerated rather than quoted.

In [ ]:
selection = load_selection()
X = build_features(tx, signals, families=summary['families'])[selection['columns']]
y = signals[config.TARGET]
print('feature matrix:', X.shape)

params = summary['tuned_params']['lightgbm']
result = evaluate(make_factory('lightgbm', params), X, y)
print(f'LightGBM repeated CV: mean={result.mean:.5f} std={result.std:.5f} score={result.score:.5f}')

## What the model actually uses

In [ ]:
import shap

model = make_factory('lightgbm', params)(config.SEED)
model.fit(X.fillna(X.median()), y)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X.fillna(X.median()))
shap.summary_plot(shap_values, X.fillna(X.median()), max_display=20, show=True)

## Submission

The prediction file is produced by `python -m src.pipeline`, which refuses to write
unless every competition rule passes. The cell below reproduces it from the frozen
selection and tuned parameters — no new search — and prints the rule check.

In [ ]:
if config.TEST_TX.exists():
    from src.pipeline import predict_from_saved
    try:
        predict_from_saved()
    except Exception as exc:
        print('Submission step could not run:', exc)
else:
    print('test_transactions.parquet is absent; run `python -m src.pipeline` once it is in place.')

## Conclusion

A compact, measured feature set with a regularized, seed-bagged ensemble ranks
alerts better than a larger feature set does. The discipline that mattered was
honest validation: on 14,000 rows with this little signal, a difference smaller
than the repeat-to-repeat standard deviation is not an improvement, and treating
it as one is how a model that looks good locally fails on the leaderboard.

Everything above is reproducible from `src/` with `SEED = 42`.